# Weather Transform Helper (Legacy)

**Source script:** `weather_data_transform.py`

Notebook mirror for submission traceability.

In [ ]:
import os
import re
import pandas as pd
from datetime import datetime




CATS_XLSX = "Data_set_cats for analysis.xlsx"
CATS_SHEET = "Tabelle1"

CACHE_DIR = "cache_openmeteo"
MANIFEST_PATH = os.path.join(CACHE_DIR, "manifest.csv")
OUT_CSV = "analysis_dataset.csv"
OUT_XLSX = "analysis_dataset.xlsx"

ID_COL = "ID"
ZIP_COL = "zip code"
DATE_COL = "date of presentation"


KEEP_ONLY_THESE_WEATHER_PREFIXES = None










def to_zip5(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()

    s = re.sub(r"\D+", "", s)
    return s.zfill(5)

def to_date(x) -> pd.Timestamp:

    return pd.to_datetime(x).normalize()

def load_manifest(path: str) -> pd.DataFrame:
    m = pd.read_csv(path)

    m["zip"] = m["zip"].astype(str).str.zfill(5)
    m["start_date"] = pd.to_datetime(m["start_date"]).dt.normalize()
    m["end_date"] = pd.to_datetime(m["end_date"]).dt.normalize()
    return m

def pick_job_for(zip5: str, date: pd.Timestamp, manifest: pd.DataFrame):
    """Find manifest row whose zip matches and date is inside [start_date, end_date]."""
    hits = manifest[(manifest["zip"] == zip5) & (manifest["start_date"] <= date) & (manifest["end_date"] >= date)]
    if hits.empty:
        return None

    hits = hits.copy()
    hits["range_days"] = (hits["end_date"] - hits["start_date"]).dt.days
    return hits.sort_values("range_days").iloc[0]

def friendly_label(col: str) -> tuple[str, str]:
    """
    Returns (human_label, unit_guess).
    Unit guesses are Open-Meteo typical defaults; verify once if you need to be 100% strict.
    """
    unit_guess = ""
    base = col
    suffix = ""


    for sfx in ["__day1", "__mean_2d", "__mean_3d", "__mean_7d", "__mean_13d", "__peak_7d", "__peak_14d"]:
        if col.endswith(sfx):
            base = col[: -len(sfx)]
            suffix = sfx.replace("__", "")
            break


    if base.startswith("temperature_2m") or base.startswith("apparent_temperature"):
        unit_guess = "°C"
    elif base in ["precipitation_sum", "rain_sum"]:
        unit_guess = "mm"
    elif base == "precipitation_hours":
        unit_guess = "hours"
    elif base == "snowfall_sum":
        unit_guess = "cm (Open-Meteo default)"
    elif base.startswith("wind_speed") or base.startswith("wind_gusts"):
        unit_guess = "km/h (Open-Meteo default)"
    elif base.startswith("wind_direction"):
        unit_guess = "°"
    elif base == "shortwave_radiation_sum":
        unit_guess = "MJ/m² (Open-Meteo default)"
    elif base == "et0_fao_evapotranspiration":
        unit_guess = "mm"
    elif base == "uv_index_max":
        unit_guess = "index"
    elif base.startswith("surface_pressure"):
        unit_guess = "hPa"
    elif base in ["sunrise", "sunset"]:
        unit_guess = "timestamp (local)"
    elif base == "daylight_duration" or base == "sunshine_duration":
        unit_guess = "seconds"


    base_pretty = base.replace("_", " ")

    if suffix == "day1":
        suffix_pretty = "on presentation date"
    elif suffix.startswith("mean_"):
        d = suffix.split("_")[1]
        suffix_pretty = f"mean over previous {d} days (ending on date)"
    elif suffix.startswith("peak_"):
        d = suffix.split("_")[1]
        suffix_pretty = f"max over previous {d} days (ending on date)"
    else:
        suffix_pretty = ""

    human = f"{base_pretty} — {suffix_pretty}".strip(" —")
    return human, unit_guess

def build_dictionary(weather_cols: list[str]) -> pd.DataFrame:
    rows = []
    for c in weather_cols:
        human, unit = friendly_label(c)
        rows.append({
            "column": c,
            "human_label": human,
            "unit": unit,
            "definition": (
                "day1 = value on date; mean_Xd = rolling mean over X days ending on date; "
                "peak_Xd = rolling max over X days ending on date."
            )
        })

    rows.append({
        "column": "moon_phase",
        "human_label": "moon phase",
        "unit": "(not included)",
        "definition": "Requested, but Open-Meteo Archive API rejected variable; pipeline removed it automatically. Compute separately if needed."
    })
    return pd.DataFrame(rows)




def main():
    if not os.path.exists(MANIFEST_PATH):
        raise FileNotFoundError(f"manifest not found at {MANIFEST_PATH}")

    cats = pd.read_excel(CATS_XLSX, sheet_name=CATS_SHEET)
    if ID_COL not in cats.columns or ZIP_COL not in cats.columns or DATE_COL not in cats.columns:
        raise ValueError(f"Expected columns: {ID_COL}, {ZIP_COL}, {DATE_COL}. Found: {list(cats.columns)}")

    cats = cats.copy()
    cats["zip5"] = cats[ZIP_COL].apply(to_zip5)
    cats["presentation_date"] = cats[DATE_COL].apply(to_date)

    manifest = load_manifest(MANIFEST_PATH)


    feature_cache = {}

    merged_rows = []
    missing = 0

    for idx, row in cats.iterrows():
        zip5 = row["zip5"]
        d = row["presentation_date"]

        job = pick_job_for(zip5, d, manifest)
        if job is None:
            missing += 1
            merged_rows.append({**row.to_dict(), "weather_job_id": None, "weather_missing_reason": "no_manifest_match"})
            continue

        feat_path = str(job["feature_path"])
        if not os.path.exists(feat_path):
            missing += 1
            merged_rows.append({**row.to_dict(), "weather_job_id": job["weather_job_id"], "weather_missing_reason": "feature_file_missing"})
            continue

        if feat_path not in feature_cache:
            f = pd.read_parquet(feat_path)



            if "time" in f.columns:
                if pd.api.types.is_integer_dtype(f["time"]) or pd.api.types.is_float_dtype(f["time"]):
                    f["time"] = pd.to_datetime(f["time"], origin="unix", unit="D").dt.normalize()
                else:
                    f["time"] = pd.to_datetime(f["time"]).dt.normalize()
                f = f.set_index("time")
            else:

                f.index = pd.to_datetime(f.index).normalize()

            feature_cache[feat_path] = f

        f = feature_cache[feat_path]

        if d not in f.index:
            missing += 1
            merged_rows.append({**row.to_dict(), "weather_job_id": job["weather_job_id"], "weather_missing_reason": "date_not_in_features"})
            continue

        weather = f.loc[d].to_dict()

        if KEEP_ONLY_THESE_WEATHER_PREFIXES is not None:
            weather = {k: v for k, v in weather.items()
                       if any(k.startswith(pref + "__") for pref in KEEP_ONLY_THESE_WEATHER_PREFIXES)}

        merged_rows.append({
            **row.to_dict(),
            "weather_job_id": job["weather_job_id"],
            "weather_missing_reason": "",
            **weather
        })

    out = pd.DataFrame(merged_rows)


    front = [ID_COL, ZIP_COL, DATE_COL, "zip5", "presentation_date", "weather_job_id", "weather_missing_reason"]
    front = [c for c in front if c in out.columns]
    rest = [c for c in out.columns if c not in front]
    out = out[front + rest]


    weather_cols = [c for c in out.columns if "__" in c]
    dictionary_df = build_dictionary(weather_cols)


    out.to_csv(OUT_CSV, index=False)


    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
        out.to_excel(xw, sheet_name="Dataset", index=False)
        dictionary_df.to_excel(xw, sheet_name="Data Dictionary", index=False)

        notes = pd.DataFrame([
            {"note": "Weather features are matched by ZIP code and date of presentation (presentation_date)."},
            {"note": "For each variable: day1 = value on that date; mean_Xd/peak_Xd computed over previous X days ending on that date."},
            {"note": "Open-Meteo Archive API rejected 'moon_phase' as a daily variable; it is not included unless computed separately."},
            {"note": f"Rows with missing weather are marked in 'weather_missing_reason'. Missing count: {missing}."},
        ])
        notes.to_excel(xw, sheet_name="Notes", index=False)

    print(f"✅ Wrote {OUT_CSV}")
    print(f"✅ Wrote {OUT_XLSX}")
    print(f"Missing weather for {missing} rows (see weather_missing_reason).")

if __name__ == "__main__":
    main()
